In [1]:
# Needed if you want to run this code in Google Colab:
#%pip install mesa[rec]
#%pip install seaborn

In [2]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from dataclasses import dataclass
from typing import Any

from mesa import Model, Agent, batch_run
from mesa.space import NetworkGrid
from mesa.datacollection import DataCollector

# Set style for better visualizations
sns.set_theme()

# Infection Spreading Model - Stochastic approach

The model uses States like Safe (S), Exposed (E), Infected (I), and Recovered (R), with transitions based on probabilities (beta, epsilon, mu).

![State Images](https://asmithh.github.io/network-science-data-book/_images/compartments_SEIR.png)

In [3]:
@dataclass
class Transition:
    target: str
    prob: float
    state_dependance: str = None

@dataclass
class SpreadingState:
    name: str
    transitions: list[Transition]
    starting_amount: int = 0

In [4]:
class SpreadingAgent(Agent):
    def __init__(self, model: Model, initial_state: str, state_map: dict[str, SpreadingState]):
        super().__init__(model)
        self.state = initial_state
        self.state_map = state_map

    def step(self):
        if self.state not in self.state_map:
            raise AttributeError(f"State {self.state} not configured in state map")

        transitions = self.state_map[self.state].transitions
        for transition in transitions:
            if self.random.random() < transition.prob:
                self.state = transition.target
                break


In [5]:
class SpreadingModel(Model):
    def __init__(self, states: list[SpreadingState] | None = None, seed=123):
        super().__init__(rng=seed)

        if states is None:
            states = [
                SpreadingState(name="safe", transitions=[Transition(target="infected", prob=0.3)], starting_amount=1000),
                SpreadingState(name="infected", transitions=[Transition(target="safe", prob=0.1)], starting_amount=0),
            ]

        self.state_map = {state.name: state for state in states}

        # creating the agents
        for state in states:
            SpreadingAgent.create_agents(self, state.starting_amount, state.name, self.state_map)

        self.datacollector = DataCollector(
            model_reporters={"state_counts": SpreadingModel.sum_states}
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)
    
    # Datacollection function for counting the amount of each state
    def sum_states(self) -> dict[str, int]:
        state_counts = {}
        for agent in self.agents:
            if agent.state not in state_counts:
                state_counts[agent.state] = 0
            state_counts[agent.state] += 1
        return state_counts


In [6]:
# Visualization functions
def plot_state_over_time(data):
    """
    Plot the number of agents in each state over time using a line plot.
    """
    # Extract state counts from the dataframe
    state_counts = pd.DataFrame(data['state_counts'].tolist(), index=data.index)
    
    # Create the plot
    plt.figure(figsize=(12, 6))
    for state in state_counts.columns:
        plt.plot(state_counts.index, state_counts[state], label=state, linewidth=2, marker='o', markersize=3)
    
    plt.xlabel('Time Step', fontsize=12)
    plt.ylabel('Number of Agents', fontsize=12)
    plt.title('Agent States Over Time', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# running and analysis for one model parameter setup
INFECTION_PROBABILITY = 0.02
RECOVERY_PROBABILITY = 0.01
states = [
    SpreadingState(name="safe", transitions=[Transition(target="infected", prob=INFECTION_PROBABILITY)], starting_amount=1000),
    SpreadingState(name="infected", transitions=[Transition(target="safe", prob=RECOVERY_PROBABILITY)], starting_amount=20),
]
TIME_STEPS = 1000

model = SpreadingModel(states=states, seed=500)
model.run_for(TIME_STEPS)
data = model.datacollector.get_model_vars_dataframe()

# Visualization of the results
plot_state_over_time(data)


In [ ]:
# running and analysis for one model parameter setup
INFECTION_PROBABILITY = 0.02
RECOVERY_PROBABILITY = 0.01
states = [
    SpreadingState(name="safe", transitions=[Transition(target="infected", prob=INFECTION_PROBABILITY)], starting_amount=1000),
    SpreadingState(name="infected", transitions=[Transition(target="recovered", prob=RECOVERY_PROBABILITY)], starting_amount=20),
    SpreadingState(name="recovered", transitions=[], starting_amount=0),
]
TIME_STEPS = 1000

model = SpreadingModel(states=states, seed=500)
model.run_for(TIME_STEPS)
data = model.datacollector.get_model_vars_dataframe()

# Visualization of the results
plot_state_over_time(data)


In [ ]:
# running and analysis for one model parameter setup
EXPOSED_PROBABILITY = 0.03
INFECTION_PROBABILITY = 0.05
RECOVERY_PROBABILITY = 0.01

states = [
    SpreadingState(name="safe", transitions=[Transition(target="exposed", prob=EXPOSED_PROBABILITY)], starting_amount=1000),
    SpreadingState(name="exposed", transitions=[Transition(target="infected", prob=INFECTION_PROBABILITY)], starting_amount=40),
    SpreadingState(name="infected", transitions=[Transition(target="recovered", prob=RECOVERY_PROBABILITY)], starting_amount=40),
    SpreadingState(name="recovered", transitions=[], starting_amount=0),
]
TIME_STEPS = 500

model = SpreadingModel(states=states, seed=500)
model.run_for(TIME_STEPS)
data = model.datacollector.get_model_vars_dataframe()

# Visualization of the results
plot_state_over_time(data)


# Infection Spreading Model - Networks

Network-based models incorporate graph structures where agents are nodes and interactions are edges. Disease spread depends on network topology: high-degree nodes (hubs) accelerate transmission in scale-free networks, while random graphs show more uniform spread. Neighbor-dependent transitions model local infection risks.

In [11]:
class SpreadingNetworkAgent(Agent):
    def __init__(self, model: Model, initial_state: str, state_map: dict[str, SpreadingState], node_id: int):
        super().__init__(model)
        self.state = initial_state
        self.state_map = state_map
        self.node_id = node_id

    def step(self):
        neighbors = self.model.grid.get_neighbors(self.node_id, include_center=False)
        state_counts = {}
        for neighbor in neighbors:
            state_counts.setdefault(neighbor.state, 0)
            state_counts[neighbor.state] += 1

        transitions = self.state_map[self.state].transitions
        for transition in transitions:
            if transition.state_dependance is None:
                compare_value = 1
            else:
                compare_value = state_counts.get(transition.state_dependance, 0)
            if self.random.random() < transition.prob * compare_value:
                self.state = transition.target
                break

In [12]:
class SpreadingNetworkModel(Model):
    def __init__(self, states: list[SpreadingState] | None = None, network: nx.Graph = None, seed=123):
        super().__init__(rng=seed)

        if states is None:
            states = [
                SpreadingState(name="safe", transitions=[Transition(target="infected", prob=0.3, state_dependance="infected")], starting_amount=1000),
                SpreadingState(name="infected", transitions=[Transition(target="safe", prob=0.1)], starting_amount=0),
            ]

        if network is None:
            raise ValueError("A network graph must be provided")

        self.state_map = {state.name: state for state in states}
        self.grid = NetworkGrid(network)

        node_ids = list(self.grid.G.nodes)
        total_agents = sum(state.starting_amount for state in states)
        if len(node_ids) < total_agents:
            raise ValueError("Number of agents exceeds number of nodes in the network")

        offset = 0
        for state in states:
            assigned_nodes = node_ids[offset : offset + state.starting_amount]
            offset += state.starting_amount

            agents = SpreadingNetworkAgent.create_agents(
                self,
                state.starting_amount,
                state.name,
                self.state_map,
                node_id=assigned_nodes,
            )
            for agent in agents:
                self.grid.place_agent(agent, agent.node_id)
            
        self.datacollector = DataCollector(
            model_reporters={"state_counts": SpreadingModel.sum_states},
            agent_reporters={"state": lambda a: a.state}
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)
    
    def sum_states(self) -> dict[str, int]:
        state_counts = {}
        for agent in self.agents:
            if agent.state not in state_counts:
                state_counts[agent.state] = 0
            state_counts[agent.state] += 1
        return state_counts


In [13]:
# Visualization functions
# Define color map for states
state_colors = {
    "safe": "green",
    "exposed": "orange",
    "infected": "red",
    "recovered": "blue"
}

def plot_state_over_time(data):
    """
    Plot the number of agents in each state over time using a line plot.
    """
    # Extract state counts from the dataframe
    state_counts = pd.DataFrame(data['state_counts'].tolist(), index=data.index)
    
    # Create the plot
    plt.figure(figsize=(12, 6))
    for state in state_counts.columns:
        plt.plot(state_counts.index, state_counts[state], label=state, linewidth=2, marker='o', markersize=3, color=state_colors.get(state, 'black'))
    
    plt.xlabel('Time Step', fontsize=12)
    plt.ylabel('Number of Agents', fontsize=12)
    plt.title('Agent States Over Time', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
def plot_network_evolution(model, steps_to_plot=None, with_labels=True):
    """
    Plot the network graph at the beginning, middle, and end of the simulation,
    with nodes colored by agent state and labeled by node ID.
    """
    if steps_to_plot is None:
        max_steps = model.datacollector.get_model_vars_dataframe().index.max()
        steps_to_plot = [0, max_steps // 4, max_steps]
    
    agent_data = model.datacollector.get_agent_vars_dataframe()
    
    # Define color map for states
    state_colors = {
        "safe": "green",
        "exposed": "orange",
        "infected": "red",
        "recovered": "blue"
    }
    
    fig, axes = plt.subplots(1, len(steps_to_plot), figsize=(18, 6))
    
    for idx, step in enumerate(steps_to_plot):
        ax = axes[idx]
        
        # Get agent states at this step
        step_data = agent_data[agent_data.index.get_level_values('Step') == step].reset_index(drop=True)
        node_states = {}
        for agent_id, step_val in step_data.iterrows():
            agent = model.agents[agent_id]
            node_states[agent.node_id] = step_val['state']
        
        # Prepare node colors and labels
        node_color = [state_colors.get(node_states.get(node, "safe"), "gray") for node in model.grid.G.nodes()]
        labels = {node: str(node) for node in model.grid.G.nodes()}
        
        # Draw the graph
        pos = nx.spring_layout(model.grid.G, seed=42)  # Fixed layout for consistency
        nx.draw(
            model.grid.G, pos, ax=ax,
            node_color=node_color,
            with_labels=with_labels,
            labels=labels,
            node_size=50,
            font_size=8,
            edge_color="gray",
            alpha=0.7
        )
        
        ax.set_title(f"{idx} (Step {step})", fontsize=14, fontweight='bold')
    
    # Add legend
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=state)
                       for state, color in state_colors.items()]
    fig.legend(handles=legend_elements, loc='lower center', ncol=len(state_colors), fontsize=12)
    
    plt.tight_layout()
    plt.show()

In [ ]:
SEED = 500

# running and analysis for one model parameter setup
NUMBER_OF_AGENTS = 150
EXPOSED_PROBABILITY = 0.4
INFECTION_PROBABILITY = 0.2
RECOVERY_PROBABILITY = 0.01
TIME_STEPS = 300

GRAPH_PROBABILITY = 0.1
#graph = nx.erdos_renyi_graph(n=NUMBER_OF_AGENTS, p=GRAPH_PROBABILITY, seed=SEED)
graph = nx.barabasi_albert_graph(n=NUMBER_OF_AGENTS, m=1, seed=SEED)

states = [
    SpreadingState(name="safe", 
                   transitions=[Transition(target="exposed", prob=EXPOSED_PROBABILITY, state_dependance="infected")], 
                   starting_amount=149),
    SpreadingState(name="exposed", 
                   transitions=[Transition(target="infected", prob=INFECTION_PROBABILITY)], 
                   starting_amount=0),
    SpreadingState(name="infected", 
                   transitions=[Transition(target="recovered", prob=RECOVERY_PROBABILITY)], 
                   starting_amount=1),
    SpreadingState(name="recovered", 
                   transitions=[], 
                   starting_amount=0),
]

model = SpreadingNetworkModel(states=states, network=graph, seed=SEED)
model.run_for(TIME_STEPS)
data = model.datacollector.get_model_vars_dataframe()
agent_data = model.datacollector.get_agent_vars_dataframe()

# Visualization of the results
plot_state_over_time(data)
plot_network_evolution(model, steps_to_plot=[0, 5, 10], with_labels=False)
plot_network_evolution(model, steps_to_plot=[15, 20, 80], with_labels=False)


In [15]:
# Multigraph
def plot_multiple_state_over_time(data_frames: list[pd.DataFrame], titles: list[str] | None = None, fig_size=(14, 4)):
    """Plot one graph per dataframe in a horizontal subplot layout."""
    # Define color map for states
    state_colors = {
        "safe": "green",
        "exposed": "orange",
        "infected": "red",
        "recovered": "blue"
    }
    if titles is None:
        titles = [f"Run {i + 1}" for i in range(len(data_frames))]

    n = len(data_frames)
    fig, axes = plt.subplots(1, n, figsize=(fig_size[0] * n, fig_size[1]), squeeze=False)

    for idx, (df, title) in enumerate(zip(data_frames, titles)):
        state_counts = pd.DataFrame(df['state_counts'].tolist(), index=df.index)
        ax = axes[0, idx]

        for state in state_counts.columns:
            ax.plot(state_counts.index, state_counts[state], label=state, linewidth=2, marker='o', markersize=3, color=state_colors.get(state, 'black'))

        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Time Step', fontsize=10)
        ax.set_ylabel('Number of Agents', fontsize=10)
        ax.grid(True, alpha=0.3)
        if idx == n - 1:
            ax.legend(fontsize=9, loc='upper right')

    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive Network Evolution Visualizer using Solara
import solara

@solara.component
def NetworkEvolutionVisualizer():
    # Reactive variables for parameters
    infection_prob = solara.use_reactive(0.05)
    recovery_prob = solara.use_reactive(0.02)
    num_agents = solara.use_reactive(100)
    num_exposed = solara.use_reactive(0)
    num_infected = solara.use_reactive(1)
    time_steps = solara.use_reactive(100)
    graph_type = solara.use_reactive("barabasi_albert")  # or "erdos_renyi"
    seed = solara.use_reactive(500)
    
    # Reactive model and data
    model = solara.use_reactive(None)
    agent_data = solara.use_reactive(None)
    current_step = solara.use_reactive(0)
    
    # Function to create and run model
    def run_model():
        # Create graph
        if graph_type.value == "barabasi_albert":
            graph = nx.barabasi_albert_graph(n=num_agents.value, m=1, seed=seed.value)
        else:
            graph = nx.erdos_renyi_graph(n=num_agents.value, p=0.07, seed=seed.value)
        
        # Define states
        states = [
            SpreadingState(name="safe", 
                           transitions=[Transition(target="exposed", prob=infection_prob.value, state_dependance="infected")], 
                           starting_amount=int(num_agents.value-num_exposed.value-num_infected.value)),
            SpreadingState(name="exposed", 
                           transitions=[Transition(target="infected", prob=infection_prob.value)], 
                           starting_amount=int(num_exposed.value)),
            SpreadingState(name="infected", 
                           transitions=[Transition(target="recovered", prob=recovery_prob.value)], 
                           starting_amount=int(num_infected.value)),
            SpreadingState(name="recovered", 
                           transitions=[], 
                           starting_amount=0),
        ]
        
        # Create and run model
        new_model = SpreadingNetworkModel(states=states, network=graph, seed=seed.value)
        new_model.run_for(time_steps.value)
        
        model.set(new_model)
        agent_data.set(new_model.datacollector.get_agent_vars_dataframe())
        current_step.set(0)
    
    # Function to plot network at current step
    def plot_network():
        if model.value is None or agent_data.value is None:
            return
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Get data at current step
        step_data = agent_data.value[agent_data.value.index.get_level_values('Step') == current_step.value].reset_index(drop=True)
        node_states = {}
        for agent_id, row in step_data.iterrows():
            agent = model.value.agents[agent_id]
            node_states[agent.node_id] = row['state']
        
        # Color map
        state_colors = {"safe": "green", "exposed": "orange", "infected": "red", "recovered": "blue"}
        node_color = [state_colors.get(node_states.get(node, "safe"), "gray") for node in model.value.grid.G.nodes()]
        
        # Draw graph
        pos = nx.spring_layout(model.value.grid.G, seed=42)
        nx.draw(model.value.grid.G, pos, ax=ax, node_color=node_color, node_size=50, edge_color="gray", alpha=0.7, with_labels=False)
        
        ax.set_title(f"Network at Step {current_step.value}")
        
        # Add legend
        legend_elements = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=state)
                       for state, color in state_colors.items()]
        fig.legend(handles=legend_elements, loc='lower center', ncol=len(state_colors), fontsize=12)
    
        return fig
    
    # Step control functions
    def next_step():
        if current_step.value < time_steps.value - 1:
            current_step.set(current_step.value + 1)
    
    def run_to_end():
        current_step.set(time_steps.value - 1)
    
    def reset_to_start():
        current_step.set(0)
    
    # UI Components
    with solara.Column():
        solara.Markdown("## Interactive Network Evolution Visualizer")
        
        # Parameter controls
        with solara.Row():
            solara.Select("Graph Type", value=graph_type, values=["barabasi_albert", "erdos_renyi"])
            solara.InputInt("Number of Agents", value=num_agents)
            solara.InputInt("Time Steps", value=time_steps)
        
        with solara.Row():
            solara.InputFloat("Infected Amount", value=num_infected)
            solara.InputFloat("Exposed Amount", value=num_exposed)

        with solara.Row():
            solara.InputFloat("Infection Probability", value=infection_prob)
            solara.InputFloat("Recovery Probability", value=recovery_prob)
            solara.InputInt("Seed", value=seed)
        
        # Run button
        solara.Button("Run Simulation", on_click=run_model, color="primary")
        
        # Step control
        if model.value is not None:
            solara.Text(f"Current Step: {current_step.value}")
            with solara.Row():
                solara.Button("Next Step", on_click=next_step)
                solara.Button("Run to End", on_click=run_to_end)
                solara.Button("Reset to Start", on_click=reset_to_start)
        
        # Plot
        fig = solara.use_memo(plot_network, [current_step.value, model.value, agent_data.value])
        if fig is not None:
            solara.FigureMatplotlib(fig)

# Display the visualizer
solara.display(NetworkEvolutionVisualizer())

In [ ]:
SEED = 500

# running and analysis for one model parameter setup
NUMBER_OF_AGENTS = 1000
INFECTION_PROBABILITY = 0.02
RECOVERY_PROBABILITY = 0.05
TIME_STEPS = 1000

GRAPH_PROBABILITY = 0.01
random_graph = nx.erdos_renyi_graph(n=NUMBER_OF_AGENTS, p=GRAPH_PROBABILITY, seed=SEED)
scale_free_graph = nx.barabasi_albert_graph(n=NUMBER_OF_AGENTS, m=1, seed=SEED)

states_simple = [
    SpreadingState(name="safe", 
                   transitions=[Transition(target="exposed", prob=0.04, state_dependance="infected")], 
                   starting_amount=920),
    SpreadingState(name="exposed", 
                   transitions=[Transition(target="infected", prob=0.02)], starting_amount=40),
    SpreadingState(name="infected",  
                   transitions=[Transition(target="recovered", prob=0.008)], 
                   starting_amount=40),
    SpreadingState(name="recovered",  
                   transitions=[], 
                   starting_amount=0),
]
states_recover = [
    SpreadingState(name="safe", 
                   transitions=[Transition(target="exposed", prob=0.08, state_dependance="infected")], 
                   starting_amount=920),
    SpreadingState(name="exposed", 
                   transitions=[Transition(target="infected", prob=0.01)], 
                   starting_amount=40),
    SpreadingState(name="infected", 
                   transitions=[Transition(target="recovered", prob=0.008)], 
                   starting_amount=40),
    SpreadingState(name="recovered", 
                   transitions=[], 
                   starting_amount=0),
]
states_exposed = [
    SpreadingState(name="safe", 
                   transitions=[Transition(target="exposed", prob=0.02, state_dependance="infected")], 
                   starting_amount=920),
    SpreadingState(name="exposed", 
                   transitions=[Transition(target="infected", prob=0.05)], 
                   starting_amount=40),
    SpreadingState(name="infected", 
                   transitions=[Transition(target="recovered", prob=0.008)], 
                   starting_amount=40),
    SpreadingState(name="recovered", 
                   transitions=[], 
                   starting_amount=0),
]

parameters = {"states": [states_simple, states_recover, states_exposed], "network": [random_graph, scale_free_graph], "seed": SEED}
results = batch_run(SpreadingNetworkModel, parameters=parameters, max_steps=TIME_STEPS, data_collection_period=1, display_progress=True)

In [ ]:
results_df = pd.DataFrame(results)
display(results_df.head())

# Create one dataframe per run
run_dataframes = []
for run_id, group in results_df.groupby("RunId"):
    df = group.loc[:, ["Step", "state_counts", "network"]].reset_index(drop=True)
    run_dataframes.append(df)

# Example: inspect the first run dataframe
print("Number of runs:", len(run_dataframes))
plot_multiple_state_over_time(run_dataframes[:2], titles=[f"Run {i + 1}" for i in range(2)], fig_size=(12, 8))
plot_multiple_state_over_time(run_dataframes[2:4], titles=[f"Run {i + 1}" for i in range(2, 4)], fig_size=(12, 8))
plot_multiple_state_over_time(run_dataframes[4:], titles=[f"Run {i + 1}" for i in range(4, 6)], fig_size=(12, 8))